# Tahap 3 - Case Retrieval

Project: Case-Based Reasoning untuk Pidana Umum - Pencurian di PN Tangerang

Notebook ini digunakan sebagai bagian dari pipeline CBR.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

BASE_DIR = Path("..").resolve()

PROCESSED_DIR = BASE_DIR / "data" / "processed"
EVAL_DIR = BASE_DIR / "data" / "eval"
RESULTS_DIR = BASE_DIR / "data" / "results"

EVAL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

cases_path = PROCESSED_DIR / "cases.csv"

cases_df = pd.read_csv(cases_path, dtype=str).fillna("")

print("Jumlah kasus:", len(cases_df))
print("Kolom:", cases_df.columns.tolist())

cases_df[[
    "case_id",
    "no_perkara",
    "tanggal_putusan",
    "terdakwa",
    "amar_lainnya",
    "solution_label",
    "jumlah_kata"
]].head()

Jumlah kasus: 40
Kolom: ['case_id', 'no_perkara', 'tanggal_putusan', 'pengadilan', 'jenis_perkara', 'penuntut_umum', 'terdakwa', 'pasal', 'hakim_ketua', 'hakim_anggota', 'panitera', 'amar', 'amar_lainnya', 'catatan_amar', 'tanggal_musyawarah', 'tanggal_dibacakan', 'ringkasan_fakta', 'argumen_hukum', 'solution_text', 'solution_label', 'lama_pidana', 'sumber_url', 'raw_file', 'jumlah_kata', 'text_full']


,case_id,no_perkara,tanggal_putusan,terdakwa,amar_lainnya,solution_label,jumlah_kata
0,case_001,1022/Pid.B/2010/PN.TNG,14 Juli 2010,JAMALUDIN Bin SANIF terbukti secara sah danmey...,,Lain-lain,80
1,case_002,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,AMIN Als. UBE Bin UDIN danALIP KURNIAWAN Als. ...,,Lain-lain,322
2,case_003,1885 /Pid.B/2011/PN.TNG,15 Desember 2011,MAULANAHASANUDIN als. KEDOK binROJALI telah te...,,Lain-lain,123
3,case_004,1073/Pid.B/2019/PN Tng,10 Juli 2019,MUHAMMAD RIKI YAKUB Alias INYONG Bin RIPIN 37 ...,,Lain-lain,213
4,case_005,678/ PID.B/ 2011/ PN TNG,8 Mei 2012,di persidangan ;Telah mendengar pembacaan tunt...,,Lain-lain,342


In [2]:
def clean_text_for_retrieval(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s./-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def build_retrieval_text(row):
    parts = [
        row.get("no_perkara", ""),
        row.get("pengadilan", ""),
        row.get("jenis_perkara", ""),
        row.get("terdakwa", ""),
        row.get("pasal", ""),
        row.get("amar_lainnya", ""),
        row.get("catatan_amar", ""),
        row.get("ringkasan_fakta", ""),
        row.get("argumen_hukum", ""),
        row.get("solution_text", ""),
        row.get("text_full", "")
    ]
    
    text = " ".join([str(p) for p in parts if str(p).strip() != ""])
    
    return clean_text_for_retrieval(text)


cases_df["retrieval_text"] = cases_df.apply(build_retrieval_text, axis=1)

cases_df[["case_id", "retrieval_text"]].head()

,case_id,retrieval_text
0,case_001,1022/pid.b/2010/pn.tng pn tangerang pidana umu...
1,case_002,497 / pid.b / 2014 / pn.tng. pn tangerang pida...
2,case_003,1885 /pid.b/2011/pn.tng pn tangerang pidana um...
3,case_004,1073/pid.b/2019/pn tng pn tangerang pidana umu...
4,case_005,678/ pid.b/ 2011/ pn tng pn tangerang pidana u...


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=5000,
    min_df=1
)

tfidf_matrix = vectorizer.fit_transform(cases_df["retrieval_text"])

print("Ukuran TF-IDF matrix:", tfidf_matrix.shape)

# Simpan vectorizer dan matrix
vectorizer_path = PROCESSED_DIR / "tfidf_vectorizer.joblib"
matrix_path = PROCESSED_DIR / "tfidf_matrix.joblib"

joblib.dump(vectorizer, vectorizer_path)
joblib.dump(tfidf_matrix, matrix_path)

print("Vectorizer disimpan:", vectorizer_path)
print("TF-IDF matrix disimpan:", matrix_path)

Ukuran TF-IDF matrix: (40, 5000)
Vectorizer disimpan: /home/zack/Penalaran-Komputer-subcpmk-3/data/processed/tfidf_vectorizer.joblib
TF-IDF matrix disimpan: /home/zack/Penalaran-Komputer-subcpmk-3/data/processed/tfidf_matrix.joblib


In [4]:
def retrieve(query: str, k: int = 5):
    query_clean = clean_text_for_retrieval(query)
    
    query_vector = vectorizer.transform([query_clean])
    
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    top_indices = similarities.argsort()[::-1][:k]
    
    results = cases_df.iloc[top_indices].copy()
    results["similarity_score"] = similarities[top_indices]
    
    return results[[
        "case_id",
        "no_perkara",
        "tanggal_putusan",
        "terdakwa",
        "amar_lainnya",
        "solution_label",
        "lama_pidana",
        "similarity_score",
        "sumber_url"
    ]]

In [5]:
query = """
Terdakwa melakukan pencurian dalam keadaan memberatkan dan dijatuhi pidana penjara.
"""

results = retrieve(query, k=5)

results

,case_id,no_perkara,tanggal_putusan,terdakwa,amar_lainnya,solution_label,lama_pidana,similarity_score,sumber_url
14,case_015,1427/Pid.B/2022/PN Tng,16 Nopember 2022,HASAN FUAD Bin DAYAT 41 — 1 MENGADILI: Menyata...,,Lain-lain,,0.130270,https://putusan3.mahkamahagung.go.id/direktori...
2,case_003,1885 /Pid.B/2011/PN.TNG,15 Desember 2011,MAULANAHASANUDIN als. KEDOK binROJALI telah te...,,Lain-lain,,0.128359,https://putusan3.mahkamahagung.go.id/direktori...
31,case_032,168/Pid.B/2023/PN Tng,8 Maret 2023,PANDU ASMIARDI 57 — 4 Menyatakan Terdakwa PAND...,,Lain-lain,,0.127299,https://putusan3.mahkamahagung.go.id/direktori...
30,case_031,1686/Pid.B/2022/PN Tng,30 Nopember 2022,1.FAHRU ROJI RIZAL FAUZI als OJI Bin alm MATSO...,,Lain-lain,,0.107699,https://putusan3.mahkamahagung.go.id/direktori...
11,case_012,159/Pid.B/2022/PN Tng,15 Maret 2022,RENDI SYAPUTRA als.RENDI Bin MULYADI 24 — 0 ME...,,Lain-lain,,0.104586,https://putusan3.mahkamahagung.go.id/direktori...


In [6]:
query = """
Kasus pencurian dengan barang bukti sepeda motor dan terdakwa dijatuhi pidana penjara beberapa bulan.
"""

retrieve(query, k=5)

,case_id,no_perkara,tanggal_putusan,terdakwa,amar_lainnya,solution_label,lama_pidana,similarity_score,sumber_url
6,case_007,834 / Pid.B / 2017 / PN.TNG.,15 Juni 2017,DEDY KUMALA BIN SAHLAN bersalah secara syahdan...,,Lain-lain,,0.259295,https://putusan3.mahkamahagung.go.id/direktori...
15,case_016,1176/Pid.B/2018/PN Tng,19 Juli 2018,ISEP ISKANDAR Bin ISHAK 39 — 3 M E N G A D I L...,,Lain-lain,,0.132220,https://putusan3.mahkamahagung.go.id/direktori...
5,case_006,1527/Pid.B/2014/PN.TNG,16 September 2014,"NUR APRIYANI Binti (Alm) NURDIN, terbuktibersa...",,Lain-lain,,0.126466,https://putusan3.mahkamahagung.go.id/direktori...
36,case_037,1380 / PID.B / 2014 / PN.TNG.,28 Agustus 2014,tersebut ;Telah mendengar keterangan saksisaks...,,Lain-lain,,0.112608,https://putusan3.mahkamahagung.go.id/direktori...
2,case_003,1885 /Pid.B/2011/PN.TNG,15 Desember 2011,MAULANAHASANUDIN als. KEDOK binROJALI telah te...,,Lain-lain,,0.082023,https://putusan3.mahkamahagung.go.id/direktori...


In [7]:
demo_queries = [
    {
        "query_id": "q001",
        "query": "Terdakwa melakukan pencurian dalam keadaan memberatkan dan dijatuhi pidana penjara."
    },
    {
        "query_id": "q002",
        "query": "Kasus pencurian dengan barang bukti sepeda motor dan pidana penjara."
    },
    {
        "query_id": "q003",
        "query": "Pencurian dilakukan oleh terdakwa dan majelis hakim menjatuhkan hukuman penjara."
    },
    {
        "query_id": "q004",
        "query": "Terdakwa terbukti melakukan tindak pidana pencurian dan tetap ditahan."
    },
    {
        "query_id": "q005",
        "query": "Barang bukti dikembalikan kepada saksi korban dan terdakwa membayar biaya perkara."
    }
]

retrieval_outputs = []

for item in demo_queries:
    query_id = item["query_id"]
    query_text = item["query"]
    
    top_results = retrieve(query_text, k=5)
    
    for rank, (_, row) in enumerate(top_results.iterrows(), start=1):
        retrieval_outputs.append({
            "query_id": query_id,
            "query": query_text,
            "rank": rank,
            "case_id": row["case_id"],
            "no_perkara": row["no_perkara"],
            "similarity_score": row["similarity_score"],
            "solution_label": row["solution_label"],
            "lama_pidana": row["lama_pidana"],
            "sumber_url": row["sumber_url"]
        })

retrieval_results_df = pd.DataFrame(retrieval_outputs)

retrieval_results_path = RESULTS_DIR / "retrieval_results_sample.csv"
retrieval_results_df.to_csv(retrieval_results_path, index=False)

print("Hasil demo retrieval disimpan:")
print(retrieval_results_path)

retrieval_results_df

Hasil demo retrieval disimpan:
/home/zack/Penalaran-Komputer-subcpmk-3/data/results/retrieval_results_sample.csv


,query_id,query,rank,case_id,no_perkara,similarity_score,solution_label,lama_pidana,sumber_url
0,q001,Terdakwa melakukan pencurian dalam keadaan mem...,1,case_015,1427/Pid.B/2022/PN Tng,0.130270,Lain-lain,,https://putusan3.mahkamahagung.go.id/direktori...
1,q001,Terdakwa melakukan pencurian dalam keadaan mem...,2,case_003,1885 /Pid.B/2011/PN.TNG,0.128359,Lain-lain,,https://putusan3.mahkamahagung.go.id/direktori...
2,q001,Terdakwa melakukan pencurian dalam keadaan mem...,3,case_032,168/Pid.B/2023/PN Tng,0.127299,Lain-lain,,https://putusan3.mahkamahagung.go.id/direktori...
3,q001,Terdakwa melakukan pencurian dalam keadaan mem...,4,case_031,1686/Pid.B/2022/PN Tng,0.107699,Lain-lain,,https://putusan3.mahkamahagung.go.id/direktori...
4,q001,Terdakwa melakukan pencurian dalam keadaan mem...,5,case_012,159/Pid.B/2022/PN Tng,0.104586,Lain-lain,,https://putusan3.mahkamahagung.go.id/direktori...
5,q002,Kasus pencurian dengan barang bukti sepeda mot...,1,case_007,834 / Pid.B / 2017 / PN.TNG.,0.334783,Lain-lain,,https://putusan3.mahkamahagung.go.id/direktori...
6,q002,Kasus pencurian dengan barang bukti sepeda mot...,2,case_016,1176/Pid.B/2018/PN Tng,0.162896,Lain-lain,,https://putusan3.mahkamahagung.go.id/direktori...
7,q002,Kasus pencurian dengan barang bukti sepeda mot...,3,case_037,1380 / PID.B / 2014 / PN.TNG.,0.136568,Lain-lain,,https://putusan3.mahkamahagung.go.id/direktori...
8,q002,Kasus pencurian dengan barang bukti sepeda mot...,4,case_006,1527/Pid.B/2014/PN.TNG,0.117953,Lain-lain,,https://putusan3.mahkamahagung.go.id/direktori...
9,q002,Kasus pencurian dengan barang bukti sepeda mot...,5,case_003,1885 /Pid.B/2011/PN.TNG,0.089659,Lain-lain,,https://putusan3.mahkamahagung.go.id/direktori...


In [8]:
import json

# Ambil 10 data sebagai query uji internal
sample_cases = cases_df.sample(n=min(10, len(cases_df)), random_state=42)

eval_queries = []

for i, (_, row) in enumerate(sample_cases.iterrows(), start=1):
    query_text = row.get("ringkasan_fakta", "")
    
    if len(str(query_text).split()) < 10:
        query_text = row.get("solution_text", "")
    
    if len(str(query_text).split()) < 10:
        query_text = row.get("text_full", "")
    
    eval_queries.append({
        "query_id": f"eval_{i:03d}",
        "query": query_text,
        "ground_truth_case_id": row["case_id"],
        "ground_truth_solution_label": row["solution_label"]
    })

queries_path = EVAL_DIR / "queries.json"

with open(queries_path, "w", encoding="utf-8") as f:
    json.dump(eval_queries, f, ensure_ascii=False, indent=2)

print("queries.json berhasil dibuat:")
print(queries_path)

eval_queries[:2]

queries.json berhasil dibuat:
/home/zack/Penalaran-Komputer-subcpmk-3/data/eval/queries.json


[{'query_id': 'eval_001',
  'query': 'Pengadilan PN TANGERANG Pidana Umum Pencurian Putus : 15-05-2013 — Upload : 23-06-2014 Putusan PN TANGERANG Nomor 672 / PID.B / 2013 / PN.TNG Tanggal 15 Mei 2013 — ABDUL KHALIM Als. NARYO Bin JABUN 32 — 3 sejak tanggal 28Januari 2013 sampai dengan sekarang ;PENGADILAN NEGERI TERSEBUT ;Telah membaca berkas perkara atas nama Terdakwa tersebut ;Telah mendengar keterangan saksisaksi, dan keterangan Terdakwa ;Telah memeriksa/memperhatikan barang bukti dalam perkara tersebut ;Telah mendengar uraian tuntutan Penuntut Umum pada Kejaksaan NegeriTangerang atas diri Terdakwa, yang pada pokoknya menuntut sebagai berikut :1 Menyatakan Terdakwa ABDUL KHALIM ALS NARYO BIN JABUN bersalahmelakukan tindak pidana Pencurian Batu Ceper Kota Tangerang, atau setidaktidaknyadisalah satu tempat lain yang termasuk dalam daerah hukum Pengadilan NegeriTangerang, mengambil barang sesuatu, yang seluruhnya atau sebagian kepunyaan oranglain, dengan maksud untuk dimiliki secara me

In [9]:
print("File yang sudah dibuat:")

for path in [
    vectorizer_path,
    matrix_path,
    retrieval_results_path,
    queries_path
]:
    print(path, "=>", path.exists())

File yang sudah dibuat:
/home/zack/Penalaran-Komputer-subcpmk-3/data/processed/tfidf_vectorizer.joblib => True
/home/zack/Penalaran-Komputer-subcpmk-3/data/processed/tfidf_matrix.joblib => True
/home/zack/Penalaran-Komputer-subcpmk-3/data/results/retrieval_results_sample.csv => True
/home/zack/Penalaran-Komputer-subcpmk-3/data/eval/queries.json => True
